In [1]:
from pynq import Overlay, allocate
print("Loading bitstream...")
BITFILE = "design_1z8.bit"
ol = Overlay(BITFILE)

dma  = ol.axi_dma_0
ctrl = ol.axil_ctrla_0
audio_ip = ol.audio_proc_0

Loading bitstream...


In [ ]:
from noise import pnoise3
import numpy as np
import socket, json, time

from multiprocessing import Process
from pynq import Overlay, allocate
from numba import njit
from noise import pnoise3
import math


WIDTH  = 1280
HEIGHT = 720
BPP    = 4

SERVER_IP = "56.228.33.153"#"192.168.137.1"#
RENDER_PORT = 9003
CONTROL_PORT = 9001

def send(sock, msg):
    sock.sendall((json.dumps(msg) + "\n").encode("utf-8"))

def connect_stream(port, hello_msg):
    while True:
        try:
            s = socket.create_connection((SERVER_IP, port), timeout=10)
            s.settimeout(None)
            f = s.makefile("r")
            send(s, hello_msg)
            print("ack:", f.readline().strip())
            return s, f
        except Exception as e:
            print(f"reconnecting port {port}...", e)
            time.sleep(1)

def clamp(v, lo=-1, hi=1):
    return max(lo, min(hi, v))

FONT = {
    'A': [" 1 ","1 1","111","1 1","1 1"],
    'B': ["11 ","1 1","11 ","1 1","11 "],
    'C': [" 11","1  ","1  ","1  "," 11"],
    'D': ["11 ","1 1","1 1","1 1","11 "],
    'E': ["111","1  ","11 ","1  ","111"],
    'F': ["111","1  ","11 ","1  ","1  "],
    'G': [" 11","1  ","1 1","1 1"," 11"],
    'H': ["1 1","1 1","111","1 1","1 1"],
    'I': ["111"," 1 "," 1 "," 1 ","111"],
    'J': [" 11","  1","  1","1 1"," 1 "],
    'K': ["1 1","1 1","11 ","1 1","1 1"],
    'L': ["1  ","1  ","1  ","1  ","111"],
    'M': ["1 1","111","111","1 1","1 1"],
    'N': ["1 1","111","111","111","1 1"],
    'O': ["111","1 1","1 1","1 1","111"],
    'P': ["111","1 1","111","1  ","1  "],
    'Q': ["111","1 1","1 1","111","  1"],
    'R': ["111","1 1","111","1 1","1 1"],
    'S': [" 11","1  ","111","  1","11 "],
    'T': ["111"," 1 "," 1 "," 1 "," 1 "],
    'U': ["1 1","1 1","1 1","1 1","111"],
    'V': ["1 1","1 1","1 1","1 1"," 1 "],
    'W': ["1 1","1 1","111","111","1 1"],
    'X': ["1 1","1 1"," 1 ","1 1","1 1"],
    'Y': ["1 1","1 1"," 1 "," 1 "," 1 "],
    'Z': ["111","  1"," 1 ","1  ","111"],

    '0': ["111","1 1","1 1","1 1","111"],
    '1': [" 1 ","11 "," 1 "," 1 ","111"],
    '2': ["111","  1","111","1  ","111"],
    '3': ["111","  1","111","  1","111"],
    '4': ["1 1","1 1","111","  1","  1"],
    '5': ["111","1  ","111","  1","111"],
    '6': ["111","1  ","111","1 1","111"],
    '7': ["111","  1"," 1 "," 1 "," 1 "],
    '8': ["111","1 1","111","1 1","111"],
    '9': ["111","1 1","111","  1","111"],

    ':': ["   "," 1 ","   "," 1 ","   "],
    '.': ["   ","   ","   "," 1 ","   "],
    '-': ["   ","   ","111","   ","   "],
    '+': ["   "," 1 ","111"," 1 ","   "],
    ' ': ["   ","   ","   ","   ","   "],
}

def draw_char(fb, x, y, ch, color=(255,255,255), scale=1):
    glyph = FONT.get(ch.upper(), FONT[' '])
    for j, row in enumerate(glyph):
        for i, c in enumerate(row):
            if c == '1':
                for dy in range(scale):
                    for dx in range(scale):
                        px = x + i*scale + dx
                        py = y + j*scale + dy

                        if 0 <= px < WIDTH and 0 <= py < HEIGHT:
                            idx = (py * WIDTH + px) * 4
                            fb[idx+0] = color[2]
                            fb[idx+1] = color[1]
                            fb[idx+2] = color[0]
                            fb[idx+3] = 255
                            
def draw_text(fb, x, y, text, scale=1, spacing=1):
    cx = x
    for ch in text:
        draw_char(fb, cx, y, ch, scale=scale)
        cx += (3*scale) + spacing*scale
        
def fbm(x, y, z, octaves=3):
    value = 0.0
    amp = 0.5
    freq = 1.0
    max_amp = 0.0

    for _ in range(octaves):
        value += pnoise3(x*freq, y*freq, z*freq) * amp
        max_amp += amp
        amp *= 0.5
        freq *= 2.1

    return value / max_amp
def generate_asteroid(subdivisions=2, noise_scale=1.8, noise_amp=0.3, scale = 1):

    t = (1 + np.sqrt(5)) / 2

    verts = [
        [-1,t,0],[1,t,0],[-1,-t,0],[1,-t,0],
        [0,-1,t],[0,1,t],[0,-1,-t],[0,1,-t],
        [t,0,-1],[t,0,1],[-t,0,-1],[-t,0,1]
    ]

    verts = [np.array(v)/np.linalg.norm(v) for v in verts]

    faces = [
        [0,11,5],[0,5,1],[0,1,7],[0,7,10],[0,10,11],
        [1,5,9],[5,11,4],[11,10,2],[10,7,6],[7,1,8],
        [3,9,4],[3,4,2],[3,2,6],[3,6,8],[3,8,9],
        [4,9,5],[2,4,11],[6,2,10],[8,6,7],[9,8,1]
    ]

    for _ in range(subdivisions):

        new_faces = []
        mid_cache = {}

        def midpoint(a,b):

            key = tuple(sorted((a,b)))
            if key in mid_cache:
                return mid_cache[key]

            v = (verts[a] + verts[b]) * 0.5
            v /= np.linalg.norm(v)

            verts.append(v)
            idx = len(verts)-1
            mid_cache[key] = idx

            return idx

        for a,b,c in faces:

            ab = midpoint(a,b)
            bc = midpoint(b,c)
            ca = midpoint(c,a)

            new_faces += [
                [a,ab,ca],
                [b,bc,ab],
                [c,ca,bc],
                [ab,bc,ca]
            ]

        faces = new_faces

    verts = np.array(verts)
    faces = np.array(faces, dtype=np.int32)

    for i,v in enumerate(verts):

        n = fbm(
            v[0]*noise_scale,
            v[1]*noise_scale,
            v[2]*noise_scale,
            3
        )
        verts[i] = v * (1 + n * noise_amp * 0.2)*scale

    return verts,faces


def audio_loop(base_addr):

    from pynq import MMIO
    import time

    audio = MMIO(base_addr, 0x1000)

    audio.write(0x00, 0xFFFF)

    print("Audio process started")

    last_clap_start = 0
    last_tone_start = 0
    TIME_THRESH = 0.2
    # audio control stream
    control_sock, _ = connect_stream(
        CONTROL_PORT, {"type": "hello", "role": "control_lr", "node_id": "pynq-lr"}
    )
    lr_value = 0
    last_lr = 0
    while True:

        irq = audio.read(7*4)

        if irq:

            data = [audio.read(i*4) for i in range(8)]
            audio.write(7*4, 0)

            total = data[0]+data[1]+data[2]+data[3]
            if total == 0:
                continue

            ratio = data[4]/total

            if data[4] > 80000:

                now = time.time()

                new_clap = now-last_clap_start > TIME_THRESH
                new_tone = now-last_tone_start > TIME_THRESH

                if (data[0] > 1000 and data[1] > 2000 and data[3] > 60 and data[4] > 60 and ratio < 75):

                    last_clap_start = now

                    if new_clap:
                        print("clap")

                elif data[4]/max(data[0],1) > 100 and data[5] > 15 and ratio > 100:

                    last_tone_start = now
                    #18-26 high
                    #10-15 low

                    if new_tone:
                        print("tone", data[5])
                        lr_value = 1 if data[5] > 20 else -1
        if lr_value !=0 or last_lr != 0:
            if last_lr != 0:
                lr_value = 0
            last_lr = lr_value
            try:
                #Audio controls up/down
                now = time.time()
                send(
                    control_sock,
                    {"type": "control", "axis": "ud", "value": lr_value, "t": now},
                )
            except Exception:
                try:
                    control_sock.close()
                except Exception:
                    pass
                control_sock, _ = connect_stream(
                    CONTROL_PORT,
                    {"type": "hello", "role": "control_lr", "node_id": "pynq-lr"},
                )

        time.sleep(0.02)

def generate_ship_mesh(scale=1.0):
    #Predefined vertices from blender
    verts = np.array([
        ( 0.000,  0.000,  1.800),  # 0  nose tip

        # Ring A — just behind nose
        ( 0.000,  0.130,  1.200),  # 1  top
        ( 0.120, -0.060,  1.200),  # 2  right
        ( 0.000, -0.130,  1.200),  # 3  bottom
        (-0.120, -0.060,  1.200),  # 4  left

        # Ring B — mid-forward
        ( 0.000,  0.300,  0.500),  # 5  top
        ( 0.340, -0.080,  0.500),  # 6  right
        ( 0.000, -0.260,  0.500),  # 7  bottom
        (-0.340, -0.080,  0.500),  # 8  left

        # Ring C — mid
        ( 0.000,  0.280, -0.100),  # 9  top
        ( 0.380, -0.100, -0.100),  # 10 right
        ( 0.000, -0.260, -0.100),  # 11 bottom
        (-0.380, -0.100, -0.100),  # 12 left

        # Ring D — rear
        ( 0.000,  0.200, -0.800),  # 13 top
        ( 0.240, -0.080, -0.800),  # 14 right
        ( 0.000, -0.200, -0.800),  # 15 bottom
        (-0.240, -0.080, -0.800),  # 16 left

        # Ring E — tail
        ( 0.000,  0.130, -1.400),  # 17 top
        ( 0.150, -0.050, -1.400),  # 18 right
        ( 0.000, -0.130, -1.400),  # 19 bottom
        (-0.150, -0.050, -1.400),  # 20 left

        # Tail cap centre
        ( 0.000,  0.000, -1.500),  # 21

        # Cockpit ridge
        ( 0.000,  0.520,  0.450),  # 22 front peak
        ( 0.000,  0.480,  0.050),  # 23 rear peak

        # Right wing
        ( 0.380, -0.060,  0.400),  # 24 root LE
        ( 0.380, -0.080, -0.650),  # 25 root TE
        ( 1.500, -0.120,  0.050),  # 26 tip LE
        ( 1.350, -0.150, -0.850),  # 27 tip TE
        ( 0.900, -0.180, -1.050),  # 28 tip inner TE

        # Left wing
        (-0.380, -0.060,  0.400),  # 29
        (-0.380, -0.080, -0.650),  # 30
        (-1.500, -0.120,  0.050),  # 31
        (-1.350, -0.150, -0.850),  # 32
        (-0.900, -0.180, -1.050),  # 33

        # Right nacelle
        ( 0.820,  0.050, -0.500),  # 34 intake top-right
        ( 0.960, -0.140, -0.500),  # 35 intake bottom
        ( 0.650, -0.100, -0.500),  # 36 intake top-left
        ( 0.820,  0.050, -1.250),  # 37 nozzle top-right
        ( 0.960, -0.140, -1.250),  # 38 nozzle bottom
        ( 0.650, -0.100, -1.250),  # 39 nozzle top-left
        ( 0.810, -0.060, -1.350),  # 40 nozzle cap centre

        # Left nacelle
        (-0.820,  0.050, -0.500),  # 41
        (-0.960, -0.140, -0.500),  # 42
        (-0.650, -0.100, -0.500),  # 43
        (-0.820,  0.050, -1.250),  # 44
        (-0.960, -0.140, -1.250),  # 45
        (-0.650, -0.100, -1.250),  # 46
        (-0.810, -0.060, -1.350),  # 47
     ])

    faces = np.array([
        # Nose cone (4)
        (0, 1, 2), (0, 2, 3), (0, 3, 4), (0, 4, 1),

        # Fuselage A→B
        (1, 2, 5), (2, 6, 5),
        (2, 3, 6), (3, 7, 6),
        (3, 4, 7), (4, 8, 7),
        (4, 1, 8), (1, 5, 8),

        # Fuselage B→C
        (5, 6, 9), (6, 10, 9),
        (6, 7, 10), (7, 11, 10),
        (7, 8, 11), (8, 12, 11),
        (8, 5, 12), (5, 9, 12),

        # Fuselage C→D
        (9, 10, 13), (10, 14, 13),
        (10, 11, 14), (11, 15, 14),
        (11, 12, 15), (12, 16, 15),
        (12, 9, 16), (9, 13, 16),

        # Fuselage D→E
        (13, 14, 17), (14, 18, 17),
        (14, 15, 18), (15, 19, 18),
        (15, 16, 19), (16, 20, 19),
        (16, 13, 20), (13, 17, 20),

        # Tail cap (4)
        (17, 18, 21), (18, 19, 21), (19, 20, 21), (20, 17, 21),

        # Cockpit bump
        (5, 22, 9), (22, 23,  9), (23,  9, 10),
        (5, 9, 22), (22, 9,  23),

        # Right wing
        (24, 25, 26), (26, 25, 27),
        (25, 27, 28),
        (24, 26, 28), (24, 28, 25),
            
        (24, 26, 25), (26, 27, 25),
        (25, 28, 27),
        (24, 28, 26), (24, 25, 28),

        # Left wing
        (29, 30, 31), (31, 30, 32),
        (30, 32, 33),
        (29, 31, 33), (29, 33, 30),
            
        (29, 31, 30), (31, 32, 30),
        (30, 33, 32),
        (29, 33, 31), (29, 30, 33),

        # Right nacelle body
            
        (34, 36, 35),  # intake cap

        (34, 35, 37),
        (35, 38, 37),

        (35, 36, 38),
        (36, 39, 38),

        (36, 34, 39),
        (34, 37, 39),
        # Right nozzle cap
        (37, 38, 40), (38, 39, 40), (39, 37, 40),

        # Left nacelle body
        (41, 42, 43),  # intake cap

        (41, 44, 42),
        (42, 44, 45),

        (42, 45, 43),
        (43, 45, 46),

        (43, 46, 41),
        (41, 46, 44),
        # Left nozzle cap
        (45, 44,  47), (46, 45,  47), (44, 46,  47),
    ], dtype=np.int32)
 
    #vertex colours
    vcols = np.array([
        0x7a8a79,  # 0  nose tip

        # Ring A — just behind nose
        0x5b7859,  # 1  top
        0x5b7859,  # 2  right
        0x5b7859,  # 3  bottom
        0x5b7859,  # 4  left

        # Ring B — mid-forward
        0x4c6e49,  # 5  top
        0x4c6e49,  # 6  right
        0x4c6e49,  # 7  bottom
        0x4c6e49,  # 8  left

        # Ring C — mid
        0x42753d,  # 9  top
        0x42753d,  # 10 right
        0x42753d,  # 11 bottom
        0x42753d,  # 12 left

        # Ring D — rear
        0x275723,  # 13 top
        0x275723,  # 14 right
        0x275723,  # 15 bottom
        0x275723,  # 16 left

        # Ring E — tail
        0x3d8a37,  # 17 top
        0x3d8a37,  # 18 right
        0x3d8a37,  # 19 bottom
        0x3d8a37,  # 20 left

        # Tail cap centre
        0x57bf4e,  # 21

        # Cockpit ridge
        0x66ba86,  # 22 front peak
        0x66ba86,  # 23 rear peak

        # Right wing
        0x286934,  # 24 root LE
        0x286934,  # 25 root TE
        0x286934,  # 26 tip LE
        0x286934,  # 27 tip TE
        0x286934,  # 28 tip inner TE

        # Left wing
        0x286934,  # 24 root LE
        0x286934,  # 25 root TE
        0x286934,  # 26 tip LE
        0x286934,  # 27 tip TE
        0x286934,  # 28 tip inner TE

        # Right nacelle
        0x57bf4e,  # 34 intake top-right
        0x57bf4e,  # 35 intake bottom
        0x57bf4e,  # 36 intake top-left
        0x57bf4e,  # 37 nozzle top-right
        0x57bf4e,  # 38 nozzle bottom
        0x57bf4e,  # 39 nozzle top-left
        0xCC0000,  # 40 nozzle cap centre

        # Left nacelle
        0x57bf4e,  # 34 intake top-right
        0x57bf4e,  # 35 intake bottom
        0x57bf4e,  # 36 intake top-left
        0x57bf4e,  # 37 nozzle top-right
        0x57bf4e,  # 38 nozzle bottom
        0x57bf4e,  # 39 nozzle top-left
        0xCC0000,  # 40 nozzle cap centre
     ])
    
    vert_col = []
    
    for f in faces:
        t = [0,0,0]
        for i in range(3):
            t[i] = (vcols[f[i]])
        vert_col.append(t)


    vert_col = np.array(vert_col)

    verts *= scale

    return verts, faces, vert_col

@njit
def pack_light(nx,ny,nz):
    return ((nz & 0xFF) << 16) | ((ny & 0xFF) << 8) | (nx & 0xFF)

@njit
def perspective(fov, aspect, near, far):
    f = 1/np.tan(fov/2)
    m = np.zeros((4,4))
    m[0,0] = f/aspect
    m[1,1] = f
    m[2,2] = (far+near)/(near-far)
    m[2,3] = (2*far*near)/(near-far)
    m[3,2] = -1
    return m

@njit
def lookat(eye,target,up):
    f = target-eye
    f = f/np.linalg.norm(f)
    s = np.cross(f,up)
    s = s/np.linalg.norm(s)
    u = np.cross(s,f)
    m = np.identity(4)
    m[0,:3] = s
    m[1,:3] = u
    m[2,:3] = -f
    t = np.identity(4)
    t[:3,3] = -eye
    return m @ t

@njit
def rotation_x(a):
    c = np.cos(a)
    s = np.sin(a)
    m = np.identity(4)
    m[1,1] = c
    m[1,2] = -s
    m[2,1] = s
    m[2,2] = c
    return m

@njit
def rotation_y(a):
    c = np.cos(a)
    s = np.sin(a)
    m = np.identity(4)
    m[0,0] = c
    m[0,2] = s
    m[2,0] = -s
    m[2,2] = c
    return m

@njit
def rotation_z(a):
    c = np.cos(a)
    s = np.sin(a)
    m = np.identity(4)
    m[0,0] = c
    m[0,1] = -s
    m[1,0] = s
    m[1,1] = c
    return m
@njit
def scale_matrix(s):
    m = np.identity(4)
    m[0,0] = s
    m[1,1] = s
    m[2,2] = s
    return m
@njit
def translation(x,y,z):
    m = np.identity(4)
    m[0,3] = x
    m[1,3] = y
    m[2,3] = z
    return m

@njit
def build_triangles(
        cols,
        faces,
        sx, sy, sz,
        world,
        normals,
        light,
        out
    ):
    idx = 0

    for i in range(faces.shape[0]):

        f0 = faces[i,0]
        f1 = faces[i,2]
        f2 = faces[i,1]

        sx0 = sx[f0]
        sy0 = sy[f0]
        sz0 = sz[f0]

        sx1 = sx[f1]
        sy1 = sy[f1]
        sz1 = sz[f1]

        sx2 = sx[f2]
        sy2 = sy[f2]
        sz2 = sz[f2]
        
        area = (np.int32(sx1) - np.int32(sx0))*(np.int32(sy2) - np.int32(sy0)) - (np.int32(sy1) - np.int32(sy0))*(np.int32(sx2) - np.int32(sx0))
        #back-face culling
        if area < 0:
            continue
        
        pos0 = (np.uint64(sz0) << np.uint64(32)) | (np.uint64(sy0) << np.uint64(20)) | (np.uint64(sx0) << np.uint64(4))
        pos1 = (np.uint64(sz1) << np.uint64(32)) | (np.uint64(sy1) << np.uint64(20)) | (np.uint64(sx1) << np.uint64(4))
        pos2 = (np.uint64(sz2) << np.uint64(32)) | (np.uint64(sy2) << np.uint64(20)) | (np.uint64(sx2) << np.uint64(4))

        n = normals[i]

        nx = int(n[0]*(1<<6)) & 0xFF
        ny = int(n[1]*(1<<6)) & 0xFF
        nz = int(n[2]*(1<<6)) & 0xFF

        normal = (nz<<16)|(ny<<8)|nx
        #currently unused
        wv0 = world[f0]
        wv1 = world[f1]
        wv2 = world[f2]
        wpos0 = np.uint64(0)#(np.uint64(int(wv0[2]*256))<<32)|(np.uint64(int(wv0[1]*256))<<16)|np.uint64(int(wv0[0]*256))
        wpos1 = np.uint64(0)#(np.uint64(int(wv1[2]*256))<<32)|(np.uint64(int(wv1[1]*256))<<16)|np.uint64(int(wv1[0]*256))
        wpos2 = np.uint64(0)#(np.uint64(int(wv2[2]*256))<<32)|(np.uint64(int(wv2[1]*256))<<16)|np.uint64(int(wv2[0]*256))

        out[idx+0] = np.uint64(pos0)
        out[idx+1] = np.uint64(pos1)
        out[idx+2] = np.uint64(pos2)
        out[idx+3] = np.uint64(cols[i][0]) | (np.uint64(cols[i][1]) << np.uint64(32))
        out[idx+4] = np.uint64(cols[i][2]) | (np.uint64(normal) << np.uint64(32))
        out[idx+5] = wpos0
        out[idx+6] = wpos1 | (((wpos2 >> np.uint64(32)) & np.uint64(0xFFFF)) << np.uint64(48))
        out[idx+7] = (wpos2 & np.uint64(0xFFFFFFFF)) | (np.uint64(light) << np.uint64(32))

        idx += 8

    return idx

@njit
def create_asteroid(x,y,z,ax,ay,az,size, verts, faces, cols, VP):
    aspect = WIDTH/HEIGHT
    model = (
        translation(x,y,z) @
        rotation_z(az) @
        rotation_y(ay) @
        rotation_x(ax)
    )
    model = model @ scale_matrix(size)

    MVP = VP @ model

    light_d = np.array([0.5,0.2,0.4])
    light_d /= np.linalg.norm(light_d)

    lx = int(light_d[0]*(1<<6)) & 0xFF
    ly = int(light_d[1]*(1<<6)) & 0xFF
    lz = int(light_d[2]*(1<<6)) & 0xFF

    light = pack_light(lx,ly,lz)
    
    n = verts.shape[0]
    verts_h = np.empty((n,4), dtype=np.float64)

    verts_h[:,0] = verts[:,0]
    verts_h[:,1] = verts[:,1]
    verts_h[:,2] = verts[:,2]
    verts_h[:,3] = 1.0

    world = (model @ verts_h.T).T

    clip = (MVP @ verts_h.T).T
    ndc = clip[:,:3] / clip[:,3:4]

    sx = ((ndc[:,0]*0.5+0.5)*WIDTH).astype(np.int32)
    sy = ((1-(ndc[:,1]*0.5+0.5))*HEIGHT).astype(np.int32)
    sz = ((ndc[:,2]*0.5+0.5)*65535).astype(np.int32)

    sx = np.clip(sx,0,WIDTH-1)
    sy = np.clip(sy,0,HEIGHT-1)
    sz = np.clip(sz,0,0xFFFF)

    v0 = verts[faces[:,0]]
    v1 = verts[faces[:,2]]
    v2 = verts[faces[:,1]]

    normals = np.cross(v1-v0, v2-v0) * -1
    normals = normals @ model[:3,:3].T
    
    for i in range(normals.shape[0]):
        nx = normals[i,0]
        ny = normals[i,1]
        nz = normals[i,2]

        l = np.sqrt(nx*nx + ny*ny + nz*nz)

        if l != 0.0:
            normals[i,0] = nx / l
            normals[i,1] = ny / l
            normals[i,2] = nz / l


    out = np.empty(len(faces)*8, dtype=np.uint64)

    count = build_triangles(
        cols,
        faces,
        sx,sy,sz,
        world,
        normals,
        light,
        out
    )
    

    return out[:count]


@njit
def render_asteroids_v(asteroids, verts, faces,cols, ship_verts, ship_faces, ship_cols, proj, view, z_bounds):

    VP = proj @ view

    max_words = len(faces[2]) * len(asteroids) * 8
    out = np.empty(max_words, dtype=np.uint64)

    idx = 0

    for i in range(asteroids.shape[0]):
        x = asteroids[i,0]
        y = asteroids[i,1]
        z = asteroids[i,2]
        ax = asteroids[i,3]
        ay = asteroids[i,4]
        az = asteroids[i,5]
        size = asteroids[i,6]
        vtype= asteroids[i,7]
        
        j = 0
        if size>0.4 and z > z_bounds[0]:
            j=2
        elif size>0.2 and z > z_bounds[1]:
            j=1
        tris = create_asteroid(x,y,z,ax,ay,az,size,
                               ship_verts if vtype != 0 else verts[j],
                               ship_faces if vtype != 0 else faces[j],
                               ship_cols if vtype != 0 else cols[j], VP)
        n = tris.shape[0]
        out[idx:idx+n] = tris
        idx += n

    if idx >= 8:
        out[idx-1] |= (np.uint64(1) << np.uint64(63))
    return out[:idx]

print("Audio process launching")

# Start audio process using MMIO base address
audio_proc = Process(
    target=audio_loop,
    args=(audio_ip.mmio.base_addr,)
)
audio_proc.daemon = True
audio_proc.start()
#Allocate memory for the framebuffer module
FB_SIZE = WIDTH*HEIGHT*BPP
fb0 = allocate(shape=(FB_SIZE,),dtype=np.uint8)
fb1 = allocate(shape=(FB_SIZE,),dtype=np.uint8)
fb2 = allocate(shape=(FB_SIZE,),dtype=np.uint8)
fb0[:] = fb1[:] = fb2[:] = 0
fb0.flush(); fb1.flush(); fb2.flush()
#Write addresses via MMIO
ctrl.write(0x00, fb0.physical_address)
ctrl.write(0x04, fb1.physical_address)
ctrl.write(0x08, fb2.physical_address)
#Write ambient light levels
ctrl.write(0x0C, int(0.2*(1<<12)))

ast_size = 10000/(2**15-1)
ASTEROID_VERTS, ASTEROID_FACES = generate_asteroid(0, noise_scale = 0.8,noise_amp=2, scale = 1)#ast_size
MASTEROID_VERTS, MASTEROID_FACES = generate_asteroid(1, noise_scale = 0.8,noise_amp=2, scale = 1)
LASTEROID_VERTS, LASTEROID_FACES = generate_asteroid(2, noise_scale = 0.8,noise_amp=2, scale = 1)
AV = (ASTEROID_VERTS, MASTEROID_VERTS, LASTEROID_VERTS)
AF = (ASTEROID_FACES, MASTEROID_FACES, LASTEROID_FACES)
asteroid_col = (0x343434, 0x343434, 0x343434)
ASTEROID_COLS = np.array([asteroid_col for i in range(len(ASTEROID_FACES))])
MASTEROID_COLS = np.array([asteroid_col for i in range(len(MASTEROID_FACES))])
LASTEROID_COLS = np.array([asteroid_col for i in range(len(LASTEROID_FACES))])
AC = (ASTEROID_COLS,MASTEROID_COLS,LASTEROID_COLS)

MAX_TRIS = 8184
MAX_WORDS = MAX_TRIS * 8

SHIP_VERTS, SHIP_FACES, ship_cols = generate_ship_mesh(scale=1)
SC =ship_cols
buf = allocate(shape=(MAX_WORDS,), dtype=np.uint64)

frame_time = 1.0/60.0

render_sock, render_file = connect_stream(
    RENDER_PORT, {"type": "hello", "role": "render", "node_id": "pynq-render"}
)

print("Starting render loop")
c = 0
tot = 0
tot2=0
#Perspective parameters
aspect = WIDTH / HEIGHT
fov = np.radians(62.0)
near = 0.1
far = 60.0
camera_offset = np.array([0.0, 0.35, 1.8])
camera_target_offset = np.array([0.0, 0.35, 0.0])
up = np.array([0.0, 1.0, 0.0])
proj = perspective(fov, aspect, near, far)

started = False
last_update_time = time.time()
start = time.time()
while True:

    line = render_file.readline()
    if not line:
        try:
            render_sock.close()
        except Exception:
            pass
        render_sock, render_file = connect_stream(
            RENDER_PORT,
            {"type": "hello", "role": "render", "node_id": "pynq-render"},
        )
        continue

    msg = json.loads(line)
    if msg.get("type") != "render":
        continue

    tick = msg.get("tick")
    objs = msg.get("objects", [])
    now = time.time()
    interp_dt = now - last_update_time
    interp_dt = min(interp_dt, 0.1)
    last_update_time = now

    ASTEROIDS = []

    for o in objs:
        x = o["pos"][0]
        y = o["pos"][1]
        z = o["pos"][2]
        angle = o["angle"]
        
        if o["type"] == "player":

            ASTEROIDS.append([x,y,z,0.0,math.radians(180),angle,o["size"], 1.0])
        else:
            ASTEROIDS.append([x,y,z,angle,angle*0.5,angle*0.3,o["size"], 0.0])

    ASTEROIDS = np.array(ASTEROIDS, dtype=np.float64)
    #Don't track player for now
    player_pos = [0.0,0.0,0.0]
    cam_pos = player_pos + camera_offset
    cam_target = player_pos + camera_target_offset
    view = lookat(cam_pos, cam_target, up)
     
    triangles = render_asteroids_v(ASTEROIDS, AV, AF,AC,SHIP_VERTS,SHIP_FACES,SC, proj, view, (-6.0, -9.0))

    if not started:
        #Indicate numba is done compiling
        print("Started")
        started = True
    
    total_n = triangles.shape[0]

    if total_n > 0 and not msg.get("game_over", False):
        n = triangles.shape[0]
        buf[:n] = triangles
        buf.flush()
        e2 = time.time() - start
        dma.sendchannel.transfer(buf[:total_n])
        dma.sendchannel.wait()
    #Wait for top left tiles to write before overriting with text
    time.sleep(0.0005)
    elapsed = time.time() - start
    start = time.time()
    c+=1
    tot+=elapsed
    tot2+=e2
    for fb in [fb0, fb1, fb2]:
        draw_text(fb, 20, 20, f"Score: {msg.get('score', 0)}", scale=1)
        draw_text(fb, 20, 40, f"FPS: {int(c/tot)}", scale=1)
        if msg.get("game_over", False) :
            draw_text(fb, 500, 300, "Game Over", scale=8)
            high_score = msg.get('high_score', 0)
            draw_text(fb, 520, 360, f"HIGH SCORE: {high_score}", scale=4)

    
    if elapsed < frame_time:
        time.sleep(frame_time - elapsed)

    if c>=60:
        #print(tot/c, tot2/c, total_n)
        c=0
        tot=0
        tot2=0